<a href="https://colab.research.google.com/github/namproong/HDI-knowledge-graph/blob/main/node_embed/optuna_3_trials_for_RotatE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import gc
import json
import random
import time
import traceback

import numpy as np
import pandas as pd
import torch
import optuna
import pykeen

from pykeen.triples import TriplesFactory
from pykeen.pipeline import pipeline
from pykeen.evaluation import RankBasedEvaluator
from google.colab import drive

In [ ]:
!pip install pykeen optuna

In [ ]:
!pip install pykeen

In [ ]:
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

base_path = '/content/drive/MyDrive/HDI_project_ver2'
file_path = os.path.join(base_path, 'HDI_triples_finalV2forP.parquet')

optuna_dir = os.path.join(base_path, 'optuna_outputs')
checkpoint_dir = os.path.join(optuna_dir, 'checkpoints')
os.makedirs(optuna_dir, exist_ok=True)
os.makedirs(checkpoint_dir, exist_ok=True)

RUN_TAG = "rotate_new_optuna_3V2_trial"

print("Base path:", base_path)
print("Input file:", file_path)
print("Optuna dir:", optuna_dir)
print("Checkpoint dir:", checkpoint_dir)
print("RUN_TAG:", RUN_TAG)
print("Torch version:", torch.__version__)

print("Optuna version:", optuna.__version__)

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

print("Seed set to:", SEED)
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
print("Loading data...")
t0 = time.time()

df = pd.read_parquet(file_path)
print("Raw dataframe shape:", df.shape)

df = df.iloc[:, :3].copy()
df.columns = ['head', 'relation', 'tail']
df = df.astype(str)

tf = TriplesFactory.from_labeled_triples(
    df[['head', 'relation', 'tail']].values,
    create_inverse_triples=True,
)

del df
gc.collect()

training, validation, testing = tf.split(
    ratios=[0.8, 0.1, 0.1],
    random_state=SEED,
)

print("Split complete")
print("Train triples:", training.num_triples)
print("Validation triples:", validation.num_triples)
print("Test triples:", testing.num_triples)
print("Entities:", tf.num_entities)
print("Relations (with inverse):", tf.num_relations)
print(f"Data prep time: {(time.time() - t0)/60:.2f} min")

In [ ]:
train_entities = set(training.entity_to_id.keys())
train_relations = set(training.relation_to_id.keys())

def count_unseen(factory, train_entities, train_relations):
    triples = factory.label_triples(factory.mapped_triples)
    unseen_head = sum(h not in train_entities for h, r, t in triples)
    unseen_tail = sum(t not in train_entities for h, r, t in triples)
    unseen_rel = sum(r not in train_relations for h, r, t in triples)
    return {
        "unseen_heads": unseen_head,
        "unseen_tails": unseen_tail,
        "unseen_relations": unseen_rel,
    }

valid_unseen = count_unseen(validation, train_entities, train_relations)
test_unseen = count_unseen(testing, train_entities, train_relations)

print("Validation unseen:", valid_unseen)
print("Testing unseen:", test_unseen)

In [ ]:
SEARCH_SPACE = {
    "embedding_dim": 200,
    "learning_rate_low": 1e-4,
    "learning_rate_high": 5e-3,
    "batch_size_choices": [256, 512],
    "num_epochs_hpo": 10,
}

N_TRIALS = 3

print("SEARCH_SPACE =", SEARCH_SPACE)
print("N_TRIALS =", N_TRIALS)

In [ ]:
def print_sep(char="=", n=80):
    print(char * n)

def objective(trial):
    learning_rate = trial.suggest_float(
        "learning_rate",
        SEARCH_SPACE["learning_rate_low"],
        SEARCH_SPACE["learning_rate_high"],
        log=True,
    )
    batch_size = trial.suggest_categorical(
        "batch_size",
        SEARCH_SPACE["batch_size_choices"],
    )

    checkpoint_name = f"{RUN_TAG}_trial_{trial.number}.pt"
    checkpoint_path = os.path.join(checkpoint_dir, checkpoint_name)

    # กันชนกับไฟล์เก่า
    if os.path.exists(checkpoint_path):
        os.remove(checkpoint_path)
        print(f"Deleted existing checkpoint: {checkpoint_path}")

    print_sep("=")
    print(f"START TRIAL {trial.number}")
    print(f"learning_rate = {learning_rate:.8f}")
    print(f"batch_size    = {batch_size}")
    print(f"embedding_dim = {SEARCH_SPACE['embedding_dim']}")
    print(f"num_epochs    = {SEARCH_SPACE['num_epochs_hpo']}")
    print(f"checkpoint    = {checkpoint_name}")
    print_sep("-")

    total_start = time.time()

    try:
        result = pipeline(
            training=training,
            testing=validation,   # ใช้ validation สำหรับ HPO
            model='RotatE',
            model_kwargs=dict(
                embedding_dim=SEARCH_SPACE["embedding_dim"],
            ),
            optimizer='Adam',   # <<< เพิ่มบรรทัดนี้
            optimizer_kwargs=dict(
                lr=learning_rate,
            ),
            training_kwargs=dict(
                num_epochs=SEARCH_SPACE["num_epochs_hpo"],
                batch_size=batch_size,
                checkpoint_name=checkpoint_name,
                checkpoint_directory=checkpoint_dir,
                checkpoint_frequency=0,
                checkpoint_on_failure=True,
            ),
            evaluator=RankBasedEvaluator,
            evaluator_kwargs=dict(
                filtered=True,
            ),
            random_seed=SEED,
            device=device,
        )

        mrr = result.metric_results.get_metric("both.realistic.mean_reciprocal_rank")
        hits1 = result.metric_results.get_metric("both.realistic.hits_at_1")
        hits3 = result.metric_results.get_metric("both.realistic.hits_at_3")
        hits10 = result.metric_results.get_metric("both.realistic.hits_at_10")
        elapsed = time.time() - total_start

        trial.set_user_attr("mrr", float(mrr))
        trial.set_user_attr("hits@1", float(hits1))
        trial.set_user_attr("hits@3", float(hits3))
        trial.set_user_attr("hits@10", float(hits10))
        trial.set_user_attr("elapsed_seconds", float(elapsed))
        trial.set_user_attr("checkpoint_name", checkpoint_name)

        for k, v in result.metric_results.to_flat_dict().items():
            try:
                trial.set_user_attr(k, float(v))
            except Exception:
                pass

        print(f"END TRIAL {trial.number}")
        print(f"MRR      = {mrr:.6f}")
        print(f"Hits@1   = {hits1:.6f}")
        print(f"Hits@3   = {hits3:.6f}")
        print(f"Hits@10  = {hits10:.6f}")
        print(f"Elapsed  = {elapsed/60:.2f} min")

        del result
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        print_sep("=")
        return mrr

    except RuntimeError as e:
        elapsed = time.time() - total_start
        err = str(e)

        if "out of memory" in err.lower():
            print(f"TRIAL {trial.number} PRUNED (OOM) after {elapsed/60:.2f} min")
            trial.set_user_attr("elapsed_seconds", float(elapsed))
            trial.set_user_attr("error", "OOM")
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            print_sep("=")
            raise optuna.TrialPruned()

        print(f"TRIAL {trial.number} FAILED after {elapsed/60:.2f} min")
        print("Error:", err)
        trial.set_user_attr("elapsed_seconds", float(elapsed))
        trial.set_user_attr("error", err)
        print_sep("=")
        raise

    except Exception as e:
        elapsed = time.time() - total_start
        print(f"TRIAL {trial.number} FAILED after {elapsed/60:.2f} min")
        print("Error:", str(e))
        print(traceback.format_exc())
        trial.set_user_attr("elapsed_seconds", float(elapsed))
        trial.set_user_attr("error", str(e))
        print_sep("=")
        raise

In [ ]:
study_db_path = os.path.join(optuna_dir, f"{RUN_TAG}.db")
storage_url = f"sqlite:///{study_db_path}"

study = optuna.create_study(
    study_name=RUN_TAG,
    direction="maximize",
    storage=storage_url,
    load_if_exists=False,   # เริ่มใหม่จริง
)

print("New study ready:", storage_url)
print("Study name:", RUN_TAG)

In [ ]:
import os

print("RUN_TAG =", RUN_TAG)
print("optuna_dir =", optuna_dir)

study_db_path = os.path.join(optuna_dir, f"{RUN_TAG}.db")
storage_url = f"sqlite:///{study_db_path}"

print("study_db_path =", study_db_path)
print("storage_url   =", storage_url)

print("exists /content/drive =", os.path.exists("/content/drive"))
print("exists MyDrive        =", os.path.exists("/content/drive/MyDrive"))
print("exists optuna_dir     =", os.path.exists(optuna_dir))
print("is dir optuna_dir     =", os.path.isdir(optuna_dir))
print("write access optuna_dir =", os.access(optuna_dir, os.W_OK) if os.path.exists(optuna_dir) else "dir not found")

In [ ]:
print(f"Starting NEW Optuna study: {RUN_TAG}")
print(f"Running {N_TRIALS} trials")

all_start = time.time()
study.optimize(objective, n_trials=N_TRIALS)
all_elapsed = time.time() - all_start

print(f"Optuna finished in {all_elapsed/60:.2f} min")

In [ ]:
trials_df = study.trials_dataframe()

trials_csv = os.path.join(optuna_dir, f"{RUN_TAG}_trials.csv")
trials_df.to_csv(trials_csv, index=False)

best_params = study.best_trial.params
best_params_path = os.path.join(optuna_dir, f"{RUN_TAG}_best_params.json")
with open(best_params_path, "w", encoding="utf-8") as f:
    json.dump(best_params, f, indent=2, ensure_ascii=False)

summary_cols = [
    "number",
    "state",
    "value",
    "params_learning_rate",
    "params_batch_size",
    "user_attrs_mrr",
    "user_attrs_hits@1",
    "user_attrs_hits@3",
    "user_attrs_hits@10",
    "user_attrs_elapsed_seconds",
    "user_attrs_checkpoint_name",
]

available_cols = [c for c in summary_cols if c in trials_df.columns]
summary_df = trials_df[available_cols].copy().sort_values("number")

summary_csv = os.path.join(optuna_dir, f"{RUN_TAG}_summary.csv")
summary_df.to_csv(summary_csv, index=False)

print("=" * 80)
print("FINAL HPO SUMMARY")
print("=" * 80)
print(summary_df)

print("\nBEST TRIAL")
print("Trial number:", study.best_trial.number)
print("Best MRR    :", study.best_trial.value)
print("Best params :", study.best_trial.params)
print("Best attrs  :", study.best_trial.user_attrs)

print("\nSaved full trials to:", trials_csv)
print("Saved summary to    :", summary_csv)
print("Saved best params to:", best_params_path)

In [ ]:
print(len(study.trials))